# Imports

In [1]:
import torch
from torch.utils.data import random_split, Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import r2_score

# Constants

In [2]:
device = 'cuda'
batch_size = 1

# Dataset

In [3]:
class CompoundDataset(Dataset):
    def __init__(self, X: torch.tensor, Y: torch.tensor):
        super().__init__()
        self.X = X
        self.Y = Y
        self.shape = X.shape
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.Y[index]

# Load the data

In [4]:
dataset = torch.load('data/dataset.pt')

C:\Users\agile\AppData\Local\Temp\ipykernel_37208\753396127.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load('data/dataset.pt')


# Split

In [5]:
# constants
DATASET_SIZE = len(dataset)  # Should be 499
TRAIN_RATIO = 0.9
TEST_RATIO = 1 - TRAIN_RATIO

# lengths of splits
train_len = int(TRAIN_RATIO * DATASET_SIZE)
test_len = DATASET_SIZE - train_len

torch.manual_seed(196)
train_dataset, test_dataset = random_split(dataset, [train_len, test_len])

Standardize

In [ ]:
train_idx = train_dataset.indices

X = dataset.X
Y = dataset.Y

x_mean = X[train_idx].mean(dim=0, keepdim=True)
x_std  = X[train_idx].std(dim=0, keepdim=True)

y_mean = Y[train_idx].mean(dim=0, keepdim=True)
y_std  = Y[train_idx].std(dim=0, keepdim=True)

eps = 1e-8

dataset.X = (X - x_mean) / (x_std + eps)
dataset.Y = (Y - y_mean) / (y_std + eps)    

In [16]:
print(X[:16])
print(Y[:16])

tensor([[2.0000e+00, 4.0000e+00, 2.0000e+00, 5.0000e-01, 7.0711e-01, 0.0000e+00],
        [8.0000e+00, 3.2000e+01, 3.2000e+01, 2.0000e+00, 1.0000e+00, 1.6000e+01],
        [2.8000e+01, 1.9800e+02, 3.4400e+02, 6.7619e+00, 1.5177e+00, 5.4611e+01],
        [4.8000e+01, 3.8400e+02, 7.6800e+02, 1.2000e+01, 2.1213e+00, 1.1378e+02],
        [6.2000e+01, 5.5400e+02, 1.2360e+03, 1.5319e+01, 2.3598e+00, 1.6123e+02],
        [5.4000e+01, 4.9000e+02, 1.1080e+03, 1.3319e+01, 2.0062e+00, 1.4227e+02],
        [6.2000e+01, 5.5400e+02, 1.2360e+03, 1.5319e+01, 2.3598e+00, 1.6123e+02],
        [1.8000e+01, 1.0800e+02, 1.6200e+02, 4.5000e+00, 1.2247e+00, 3.4172e+01],
        [8.6000e+01, 8.5000e+02, 2.1160e+03, 2.0944e+01, 2.9469e+00, 2.4445e+02],
        [2.0000e+00, 4.0000e+00, 2.0000e+00, 5.0000e-01, 7.0711e-01, 0.0000e+00],
        [8.0000e+01, 8.7200e+02, 2.3600e+03, 1.8022e+01, 2.6213e+00, 2.2434e+02],
        [8.0000e+00, 3.2000e+01, 3.2000e+01, 2.0000e+00, 1.0000e+00, 1.6000e+01],
        [2.0000e

Fit Diffs

In [1]:
precisions = [0.01, 0.1, 0.1, 0.01, 0.01, 0.1, 0.01, 0.1]
# precisions = [0.1, 0.1]
median_diffs = []
for col_idx in range(train_dataset.dataset.Y.shape[1]):
    column = train_dataset.dataset.Y[:, col_idx]
    column, _ = torch.sort(column)
    diffs = torch.diff(column)
    medain_diff = torch.median(diffs)
    if torch.isclose(medain_diff, torch.tensor(0.0)):
        medain_diff = precisions[col_idx]
    median_diffs.append(medain_diff)

median_diffs = torch.tensor(median_diffs)
print(median_diffs)

NameError: name 'train_dataset' is not defined

In [8]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

# Model

In [9]:
class MIC(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.fc1 = nn.Linear(input_features, 32)
        self.fc2 = nn.Linear(32, 64)
        self.fc3 = nn.Linear(64, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 16)
        self.fc6 = nn.Linear(16, output_features)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = self.fc6(x)
        
        return x
        

In [10]:
model = MIC(train_dataset.dataset.X.shape[1], train_dataset.dataset.Y.shape[1])
summary(
    model,
    train_dataset.dataset.X.shape
)

Layer (type:depth-idx)                   Output Shape              Param #
MIC                                      [152, 2]                  --
├─Linear: 1-1                            [152, 32]                 224
├─Linear: 1-2                            [152, 64]                 2,112
├─Linear: 1-3                            [152, 128]                8,320
├─Linear: 1-4                            [152, 64]                 8,256
├─Linear: 1-5                            [152, 16]                 1,040
├─Linear: 1-6                            [152, 2]                  34
Total params: 19,986
Trainable params: 19,986
Non-trainable params: 0
Total mult-adds (M): 3.04
Input size (MB): 0.00
Forward/backward pass size (MB): 0.37
Params size (MB): 0.08
Estimated Total Size (MB): 0.46

# Loss Function

In [11]:
def mic_loss(median_diffs):
    eps = 1e-6
    
    def loss_function(predicted, true):
        diffs = torch.sqrt((true - predicted)**2 + 1e-3)

        md = median_diffs.to(diffs.device)
        scaled = diffs / (md + eps)

        loss = torch.log1p(scaled).mean()
        return loss
    
    return loss_function

# Training

In [12]:
# model, optimizer, and custom loss
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-1)
# criterion = mic_loss(median_diffs)
criterion = torch.nn.MSELoss(reduction='mean')

# training parameters
num_epochs = 500
loss_ot = [] # loss overtime

In [13]:
for epoch in range(num_epochs):
    # training on training dataset
    model.train()
    total_loss = 0

    pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: Avg Training Loss = {avg_loss}")
    loss_ot.append(avg_loss)

Epoch 1/500: 100%|██████████| 16/16 [00:00<00:00, 180.22it/s]


Epoch 1: Avg Training Loss = 0.6397685709902469


Epoch 2/500: 100%|██████████| 16/16 [00:00<00:00, 430.27it/s]


Epoch 2: Avg Training Loss = 0.1937531540855108


Epoch 3/500: 100%|██████████| 16/16 [00:00<00:00, 766.85it/s]


...

Epoch 498: Avg Training Loss = 0.10462832927484722


Epoch 499/500: 100%|██████████| 16/16 [00:00<00:00, 704.16it/s]


Epoch 499: Avg Training Loss = 0.10542576732661794


Epoch 500/500: 100%|██████████| 16/16 [00:00<00:00, 730.01it/s]

Epoch 500: Avg Training Loss = 0.10138328729526085


In [14]:
y_std_d = y_std.to(device)
y_mean_d = y_mean.to(device)

for x, y in test_loader:
    x_d = x.to(device)
    y_d = y.to(device)
    pred = model(x_d)
    
    pred = (pred * y_std_d) + y_mean_d
    y_d = (y_d * y_std_d) + y_mean_d
    
    print(torch.abs(pred - y_d)[0])
    

tensor([ 1.2387, 78.9792], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([35.1387, 24.1208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([33.1387, 78.0208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([37.2613, 21.9792], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([64.1387,  7.2792], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([74.1387, 50.5208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([93.7387, 97.7208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([60.4387,  1.3208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([26.3387, 58.7208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([91.7613, 87.7208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([19.7387, 14.6792], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([ 48.7613, 125.0792], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([46.1387, 64.7208], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([ 99.3387, 148.2208], device='cuda:0', grad_fn=<SelectBackward0>)
te